# Lesson 14: Random Forests & Boosting

*ML Academy — companion notebook*

Combine many trees: bagging, random forests, AdaBoost and gradient boosting.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-14.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)

X, y = load_breast_cancer(return_X_y=True)
models = {
    "single tree": DecisionTreeClassifier(random_state=0),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=0),
    "gradient boosting": GradientBoostingClassifier(random_state=0),
    "hist. gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
for name, model in models.items():
    s = cross_val_score(model, X, y, cv=5)
    print(f"{name:24s} {s.mean():.3f} ± {s.std():.3f}")

rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0).fit(X, y)
print("out-of-bag accuracy:", round(rf.oob_score_, 3))

single tree              0.917 ± 0.016


random forest            0.963 ± 0.022


AdaBoost                 0.977 ± 0.009


gradient boosting        0.963 ± 0.021


hist. gradient boosting  0.965 ± 0.018


out-of-bag accuracy: 0.965


## 14. Going deeper: tuning ensembles and how XGBoost works

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
# max_features controls how different the trees are
for mf in [1.0, 0.5, "sqrt", 1]:
    rf = RandomForestClassifier(n_estimators=300, max_features=mf, oob_score=True, random_state=0, n_jobs=-1).fit(Xtr, ytr)
    preds = np.array([t.predict(Xte) for t in rf.estimators_])
    corr = np.corrcoef(preds)[np.triu_indices(len(preds), 1)].mean()
    print(f"max_features {str(mf):5}: tree-to-tree correlation {corr:.2f}   out-of-bag accuracy {rf.oob_score_:.3f}   test {rf.score(Xte, yte):.3f}")

max_features 1.0  : tree-to-tree correlation 0.83   out-of-bag accuracy 0.960   test 0.930


max_features 0.5  : tree-to-tree correlation 0.82   out-of-bag accuracy 0.957   test 0.942


max_features sqrt : tree-to-tree correlation 0.80   out-of-bag accuracy 0.967   test 0.953


max_features 1    : tree-to-tree correlation 0.70   out-of-bag accuracy 0.965   test 0.936


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier

for lr in [1.0, 0.3, 0.1, 0.03]:
    gb = HistGradientBoostingClassifier(learning_rate=lr, max_iter=2000, early_stopping=True,
                                        validation_fraction=0.2, n_iter_no_change=30, random_state=0).fit(Xtr, ytr)
    print(f"learning rate {lr:4}: trees used {gb.n_iter_:4d}   test accuracy {gb.score(Xte, yte):.3f}")

learning rate  1.0: trees used   35   test accuracy 0.924


learning rate  0.3: trees used   49   test accuracy 0.942


learning rate  0.1: trees used   83   test accuracy 0.947


learning rate 0.03: trees used  203   test accuracy 0.936


In [4]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

base = [("forest", RandomForestClassifier(n_estimators=300, random_state=0)),
        ("boost", HistGradientBoostingClassifier(random_state=0)),
        ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(15)))]
for name, m in base:
    print(f"{name:7}: test accuracy {m.fit(Xtr, ytr).score(Xte, yte):.3f}")
stack = StackingClassifier(base, final_estimator=LogisticRegression(max_iter=1000), cv=5).fit(Xtr, ytr)
print(f"stacked: test accuracy {stack.score(Xte, yte):.3f}   blender weights {stack.final_estimator_.coef_.round(2)}")

forest : test accuracy 0.953
boost  : test accuracy 0.942


knn    : test accuracy 0.942


stacked: test accuracy 0.942   blender weights [[2.84 2.28 3.05]]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, LeaveOneOut
cols = ["size_m2", "bedrooms", "dist_km", "age_yrs"]
for name, m in [("single tree", DecisionTreeClassifier(random_state=0)),
                ("random forest", RandomForestClassifier(n_estimators=300, random_state=0))]:
    acc = cross_val_score(m, df[cols], df["sold_fast"], cv=LeaveOneOut()).mean()
    print(f"{name:14s} leave-one-out accuracy {acc:.2f}")
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(df[cols], df["sold_fast"])
print({c: round(float(v), 2) for c, v in zip(cols, rf.feature_importances_)})

single tree    leave-one-out accuracy 0.50


random forest  leave-one-out accuracy 0.58


{'size_m2': 0.27, 'bedrooms': 0.04, 'dist_km': 0.31, 'age_yrs': 0.37}


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [6]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Majority vote

Three independent models are each 70% accurate. Compute the accuracy of their majority vote.

In [7]:
p = 0.7
vote = None  # ← your code here
check(vote, 0.7839999999999999, hint='All three right, plus exactly two right (3 ways).')

❌ Not yet. Hint: All three right, plus exactly two right (3 ways).


### Challenge 2: A bootstrap sample

Draw a bootstrap sample of indices (with replacement) and measure the share of distinct rows it contains.

In [8]:
import numpy as np
rng = np.random.default_rng(0)
n = 1000
idx = np.arange(n)  # ← replace: this takes every row exactly once, not a bootstrap sample
share_unique = len(np.unique(idx)) / n
check(share_unique, 0.622, hint='rng.integers(0, n, size=n) draws n indices with replacement. Expect about 63%.', atol=0.03)

❌ Not yet. Hint: rng.integers(0, n, size=n) draws n indices with replacement. Expect about 63%.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Majority vote** — the missing line:

```python
vote = p ** 3 + 3 * p ** 2 * (1 - p)
```

**2. A bootstrap sample** — the missing line:

```python
idx = rng.integers(0, n, size=n)
```